In [2]:
!pip install opencv-contrib-python -q

In [3]:
from google.colab import files
files.upload()

Output hidden; open in https://colab.research.google.com to view.

In [5]:
import os
import urllib.request
import glob
import cv2

In [7]:


# ──────────────────────────────────────────────
# 1. Download 4 model files nếu chưa có
# ──────────────────────────────────────────────
MODEL_DIR = "/content/wechat_models"
os.makedirs(MODEL_DIR, exist_ok=True)

BASE_URL = (
    "https://raw.githubusercontent.com/opencv/opencv_3rdparty/"
    "dnn_samples_face_detector_20170830/res10_300x300_ssd_iter_140000.caffemodel"
)

# URL chính thức từ opencv_contrib
FILES = {
    "detect.prototxt":    "https://raw.githubusercontent.com/WeChatCV/opencv_3rdparty/wechat_qrcode/detect.prototxt",
    "detect.caffemodel":  "https://raw.githubusercontent.com/WeChatCV/opencv_3rdparty/wechat_qrcode/detect.caffemodel",
    "sr.prototxt":        "https://raw.githubusercontent.com/WeChatCV/opencv_3rdparty/wechat_qrcode/sr.prototxt",
    "sr.caffemodel":      "https://raw.githubusercontent.com/WeChatCV/opencv_3rdparty/wechat_qrcode/sr.caffemodel",
}

print("=== Tải model files ===")
for filename, url in FILES.items():
    dest = os.path.join(MODEL_DIR, filename)
    if os.path.exists(dest):
        print(f"  [skip] {filename} đã tồn tại")
    else:
        print(f"  [download] {filename} ...", end=" ", flush=True)
        try:
            urllib.request.urlretrieve(url, dest)
            size_kb = os.path.getsize(dest) / 1024
            print(f"OK ({size_kb:.1f} KB)")
        except Exception as e:
            print(f"FAILED: {e}")

# ──────────────────────────────────────────────
# 2. Khởi tạo WeChatQRCode detector
# ──────────────────────────────────────────────
print("\n=== Khởi tạo WeChatQRCode ===")
try:
    detector = cv2.wechat_qrcode_WeChatQRCode(
        os.path.join(MODEL_DIR, "detect.prototxt"),
        os.path.join(MODEL_DIR, "detect.caffemodel"),
        os.path.join(MODEL_DIR, "sr.prototxt"),
        os.path.join(MODEL_DIR, "sr.caffemodel"),
    )
    print("  OK")
except AttributeError:
    raise RuntimeError(
        "cv2.wechat_qrcode không khả dụng. "
        "Cài đặt: pip install opencv-contrib-python"
    )

# ──────────────────────────────────────────────
# 3. Quét ảnh trong /content/ (không đệ quy)
# ──────────────────────────────────────────────
EXTS = ("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG")
image_paths = []
for ext in EXTS:
    image_paths.extend(glob.glob(f"/content/{ext}"))
image_paths = sorted(set(image_paths))  # dedup + sort

print(f"\n=== Tìm thấy {len(image_paths)} file ảnh trong /content/ ===\n")

if not image_paths:
    print("Không có file ảnh nào. Upload ảnh vào /content/ rồi chạy lại.")
else:
    for path in image_paths:
        filename = os.path.basename(path)
        img = cv2.imread(path)
        if img is None:
            print(f"[{filename}]  ❌ Không đọc được file")
            continue

        results, _ = detector.detectAndDecode(img)

        if results:
            for i, qr_text in enumerate(results, 1):
                label = f"QR #{i}" if len(results) > 1 else "QR"
                print(f"[{filename}]  ✅ {label}: {qr_text}")
        else:
            print(f"[{filename}]  ⚠️  Không tìm thấy mã QR")

print("\nHoàn tất.")

=== Tải model files ===
  [download] detect.prototxt ... OK (41.7 KB)
  [download] detect.caffemodel ... OK (942.8 KB)
  [download] sr.prototxt ... OK (5.8 KB)
  [download] sr.caffemodel ... OK (23.4 KB)

=== Khởi tạo WeChatQRCode ===
  OK

=== Tìm thấy 12 file ảnh trong /content/ ===

[074205007823T.jpg]  ⚠️  Không tìm thấy mã QR
[079205023503T.jpg]  ⚠️  Không tìm thấy mã QR
[080304014151T.jpg]  ✅ QR: 080304014151|301883070|Phạm Thị Trà My|22112004|Nữ|Tổ 2, Long Hưng, Long Thượng, Cần Giuộc, Long An|23032024
[089302003659T.jpg]  ✅ QR: 089302003659|352718677|Đỗ Nguyễn Tuyết Hằng|09112002|Nữ|Tổ 5, Ấp An Long, An Thạnh Trung, Chợ Mới, An Giang|07112023
[CCCD_1.jpg]  ✅ QR: 058096007531|264450733|Nguyễn Mạnh Cường|20111996|Nam|Thôn Nha Hố 1, Nhơn Sơn, Ninh Sơn, Ninh Thuận|16032022
[CCCD_2.jpg]  ✅ QR: 058096007531|264450733|Nguyễn Mạnh Cường|20111996|Nam|Thôn Nha Hố 1, Nhơn Sơn, Ninh Sơn, Ninh Thuận|16032022
[CCCD_3.jpg]  ⚠️  Không tìm thấy mã QR
[CCCD_4.jpg]  ⚠️  Không tìm thấy mã QR
[CCCD

In [1]:
!rm /content/*.jpg /content/*.jpeg /content/*.png 2>/dev/null; echo "Done"

Done
